# M5 demand forecasting

## Notebook purpose

This notebook contains the standalone machine-learning pipeline for daily item-store demand forecasting. It excludes exploratory analysis and the upstream raw M5 preprocessing performed in the team's main notebook.

- **Input:** `m5_processed.parquet`
- **Output:** `streamlit_forecast_output.csv`
- **Forecast horizon:** 28 days
- **Validation:** final 28 historical days, forecast recursively

Set the `M5_PROCESSED_PATH` environment variable or edit `PROCESSED_DATA_PATH` in the configuration cell when the Parquet file is stored elsewhere. Set `M5_OUTPUT_DIR` to change the output directory.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from lightgbm import LGBMRegressor

pd.set_option("display.max_columns", 200)

FORECAST_HORIZON = 28
SERIES_KEYS = ["item_id", "store_id"]

configured_input = os.environ.get("M5_PROCESSED_PATH")
if configured_input:
    PROCESSED_DATA_PATH = Path(configured_input)
else:
    input_candidates = [
        Path("m5_processed.parquet"),
        Path("/kaggle/working/m5_processed.parquet"),
    ]
    PROCESSED_DATA_PATH = next(
        (path for path in input_candidates if path.exists()),
        input_candidates[0],
    )

default_output_directory = (
    Path("/kaggle/working")
    if Path("/kaggle/working").exists()
    else Path(".")
)
OUTPUT_DIRECTORY = Path(
    os.environ.get("M5_OUTPUT_DIR", str(default_output_directory))
)

print("Input path:", PROCESSED_DATA_PATH)
print("Output directory:", OUTPUT_DIRECTORY)

## Load processed data

The processed Parquet file must already contain item-store sales history, calendar and event fields, state SNAP flags, and sell price. The checks below fail early if the standalone input does not match that schema.

In [ ]:
required_source_columns = [
    "item_id", "store_id", "date", "sales",
    "dept_id", "cat_id", "state_id", "d", "wm_yr_wk",
    "weekday", "wday", "month", "year",
    "event_name_1", "event_type_1", "event_name_2", "event_type_2",
    "snap_CA", "snap_TX", "snap_WI", "sell_price",
]

if not PROCESSED_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Processed dataset not found at {PROCESSED_DATA_PATH}. "
        "Set M5_PROCESSED_PATH or edit PROCESSED_DATA_PATH."
    )

demand_data = pd.read_parquet(
    PROCESSED_DATA_PATH,
    columns=required_source_columns,
)

missing_columns = set(required_source_columns) - set(demand_data.columns)
assert not missing_columns, f"Missing required columns: {sorted(missing_columns)}"

demand_data["date"] = pd.to_datetime(demand_data["date"])
demand_data = demand_data.sort_values(SERIES_KEYS + ["date"]).reset_index(drop=True)

assert not demand_data.duplicated(SERIES_KEYS + ["date"]).any(), (
    "Duplicate item-store-date rows found."
)
assert demand_data["sales"].notna().all(), "Historical sales contain missing values."
assert demand_data["sales"].ge(0).all(), "Historical sales contain negative values."

print("Dataset shape:", demand_data.shape)
print("Date range:", demand_data["date"].min(), "to", demand_data["date"].max())
print("Item-store series:", demand_data[SERIES_KEYS].drop_duplicates().shape[0])

## Forecasting data preparation

All sales lags use prior rows within the same item-store series. Rolling sales statistics use `sales.shift(1)` before rolling. The final 28 days are masked before target-derived features are created, preventing validation leakage.

In [ ]:
def grouped_rolling_stats(frame, shifted_values, window, min_periods):
    groupers = [frame[column] for column in SERIES_KEYS]
    stats = (
        shifted_values
        .groupby(groupers, sort=False, observed=True)
        .rolling(window=window, min_periods=min_periods)
        .agg(["mean", "std"])
    )
    stats.index = stats.index.droplevel(list(range(len(SERIES_KEYS))))
    return stats.reindex(frame.index)


def add_calendar_features(frame):
    frame["day_of_week"] = frame["date"].dt.dayofweek.astype("int8")
    frame["week_of_year"] = frame["date"].dt.isocalendar().week.astype("int16")
    frame["day_of_month"] = frame["date"].dt.day.astype("int8")
    frame["day_of_year"] = frame["date"].dt.dayofyear.astype("int16")
    frame["quarter"] = frame["date"].dt.quarter.astype("int8")
    frame["is_weekend"] = frame["day_of_week"].isin([5, 6]).astype("int8")
    frame["has_event_1"] = frame["event_name_1"].notna().astype("int8")
    frame["has_event_2"] = frame["event_name_2"].notna().astype("int8")
    frame["snap"] = np.select(
        [
            frame["state_id"].eq("CA"),
            frame["state_id"].eq("TX"),
            frame["state_id"].eq("WI"),
        ],
        [frame["snap_CA"], frame["snap_TX"], frame["snap_WI"]],
        default=0,
    ).astype("int8")
    return frame


def add_price_features(frame):
    price_group = frame.groupby(SERIES_KEYS, sort=False, observed=True)["sell_price"]
    frame["price_available"] = frame["sell_price"].notna().astype("int8")
    frame["price_lag_7"] = price_group.shift(7).astype("float32")

    shifted_price = price_group.shift(1)
    price_stats = grouped_rolling_stats(
        frame,
        shifted_price,
        window=28,
        min_periods=7,
    )
    frame["price_rolling_28_mean"] = price_stats["mean"].astype("float32")

    valid_lag_price = frame["price_lag_7"].replace(0, np.nan)
    valid_rolling_price = frame["price_rolling_28_mean"].replace(0, np.nan)
    frame["price_change_7"] = (
        frame["sell_price"].div(valid_lag_price).sub(1).astype("float32")
    )
    frame["price_vs_rolling_28"] = (
        frame["sell_price"].div(valid_rolling_price).sub(1).astype("float32")
    )
    frame["sell_price"] = frame["sell_price"].astype("float32")
    return frame


def add_sales_history_features(frame):
    sales_group = frame.groupby(SERIES_KEYS, sort=False, observed=True)["sales"]

    for lag in [1, 7, 14, 28]:
        frame[f"sales_lag_{lag}"] = sales_group.shift(lag).astype("float32")

    shifted_sales = sales_group.shift(1)
    for window in [7, 28]:
        stats = grouped_rolling_stats(
            frame,
            shifted_sales,
            window=window,
            min_periods=window,
        )
        frame[f"sales_rolling_{window}_mean"] = stats["mean"].astype("float32")
        frame[f"sales_rolling_{window}_std"] = stats["std"].astype("float32")

    return frame

In [ ]:
validation_end = demand_data["date"].max()
validation_start = validation_end - pd.Timedelta(days=FORECAST_HORIZON - 1)
validation_mask = demand_data["date"].ge(validation_start)

assert demand_data.loc[validation_mask, "date"].nunique() == FORECAST_HORIZON, (
    f"Expected {FORECAST_HORIZON} validation dates."
)

feature_data = demand_data.copy()
feature_data["actual_sales"] = feature_data["sales"]

# Hide all holdout targets before creating lags and rolling statistics.
feature_data.loc[validation_mask, "sales"] = np.nan

feature_data = add_calendar_features(feature_data)
feature_data = add_price_features(feature_data)
feature_data = add_sales_history_features(feature_data)

feature_data["sales"] = feature_data.pop("actual_sales")
feature_data["forecast_step"] = np.where(
    feature_data["date"].ge(validation_start),
    (feature_data["date"] - validation_start).dt.days + 1,
    0,
).astype("int8")

In [ ]:
lag_features = [
    "sales_lag_1", "sales_lag_7", "sales_lag_14", "sales_lag_28",
]
rolling_features = [
    "sales_rolling_7_mean", "sales_rolling_7_std",
    "sales_rolling_28_mean", "sales_rolling_28_std",
]
calendar_features = [
    "day_of_week", "week_of_year", "day_of_month", "day_of_year",
    "month", "quarter", "year", "is_weekend",
    "has_event_1", "has_event_2", "snap",
]
price_features = [
    "sell_price", "price_available", "price_lag_7",
    "price_rolling_28_mean", "price_change_7", "price_vs_rolling_28",
]
categorical_features = [
    "item_id", "store_id", "dept_id", "cat_id", "state_id",
    "weekday", "event_name_1", "event_type_1",
    "event_name_2", "event_type_2",
]
feature_columns = (
    categorical_features + calendar_features + price_features
    + lag_features + rolling_features
)
target_column = "sales"

for column in categorical_features:
    feature_data[column] = feature_data[column].astype("category")

train_data = feature_data.loc[
    feature_data["date"].lt(validation_start)
].copy()
validation_data = feature_data.loc[
    feature_data["date"].ge(validation_start)
].copy()

# Retain the cutoff history needed for later recursive validation forecasts.
validation_history = (
    train_data
    .groupby(SERIES_KEYS, sort=False, observed=True)
    .tail(max([1, 7, 14, 28]))
    .copy()
)

assert train_data["date"].max() < validation_data["date"].min()
assert validation_data["date"].nunique() == FORECAST_HORIZON
assert validation_data.groupby(SERIES_KEYS, observed=True).size().eq(
    FORECAST_HORIZON
).all()

# Only step 1 may use lag-1 actuals; later validation actuals were masked.
assert validation_data.loc[
    validation_data["forecast_step"].eq(1), "sales_lag_1"
].notna().all()
assert validation_data.loc[
    validation_data["forecast_step"].gt(1), "sales_lag_1"
].isna().all()

print("Training dates:", train_data["date"].min(), "to", train_data["date"].max())
print("Validation dates:", validation_data["date"].min(), "to", validation_data["date"].max())
print("Training shape:", train_data.shape)
print("Validation shape:", validation_data.shape)
print("Feature count:", len(feature_columns))

# Forecasting models

All four methods use the same recursive 28-day validation protocol. After each forecast date, predictions replace the unknown validation sales in history before the next date's lag and rolling features are recomputed. WAPE is reported as a percentage.

In [ ]:
def recursive_forecast(model_name, predictor):
    history = validation_history.copy()
    forecast_days = sorted(validation_data["date"].unique())
    daily_forecasts = []

    for forecast_date in forecast_days:
        current_day = validation_data.loc[
            validation_data["date"].eq(forecast_date)
        ].copy()
        current_day["sales"] = np.nan

        feature_frame = pd.concat(
            [history, current_day],
            ignore_index=True,
        ).sort_values(SERIES_KEYS + ["date"]).reset_index(drop=True)

        feature_frame = add_sales_history_features(feature_frame)
        current_features = feature_frame.loc[
            feature_frame["date"].eq(forecast_date)
        ].copy()

        predictions = pd.Series(
            predictor(current_features),
            index=current_features.index,
            dtype="float64",
        )
        if len(predictions) != len(current_features):
            raise ValueError(f"{model_name} returned an unexpected number of predictions.")
        if not np.isfinite(predictions).all():
            raise ValueError(f"{model_name} returned non-finite predictions.")

        predictions = predictions.clip(lower=0)

        forecast_day = current_features[SERIES_KEYS + ["date"]].copy()
        forecast_day["predicted_sales"] = predictions.to_numpy()
        daily_forecasts.append(forecast_day)

        current_features["sales"] = predictions.to_numpy()
        history = (
            pd.concat([history, current_features], ignore_index=True)
            .sort_values(SERIES_KEYS + ["date"])
            .groupby(SERIES_KEYS, sort=False, observed=True)
            .tail(28)
            .reset_index(drop=True)
        )

    forecast = pd.concat(daily_forecasts, ignore_index=True)
    forecast["Model"] = model_name
    return forecast


def calculate_forecast_metrics(actual, predicted):
    errors = actual - predicted
    absolute_errors = errors.abs()
    actual_total = actual.abs().sum()

    return {
        "MAE": absolute_errors.mean(),
        "RMSE": np.sqrt(np.mean(np.square(errors))),
        "WAPE": 100 * absolute_errors.sum() / actual_total if actual_total else np.nan,
    }

In [ ]:
baseline_predictors = {
    "Zero forecast": lambda frame: pd.Series(0.0, index=frame.index),
    "Seasonal naive (lag 7)": lambda frame: frame["sales_lag_7"],
    "Trailing 7-day moving average": lambda frame: frame["sales_rolling_7_mean"],
}

validation_forecasts = {}

for baseline_name, baseline_predictor in baseline_predictors.items():
    print(f"Generating {baseline_name}...")
    validation_forecasts[baseline_name] = recursive_forecast(
        baseline_name,
        baseline_predictor,
    )

In [ ]:
X_train = train_data.loc[:, feature_columns]
y_train = train_data.loc[:, target_column]

assert isinstance(X_train, pd.DataFrame)
assert all(
    isinstance(X_train[column].dtype, pd.CategoricalDtype)
    for column in categorical_features
)

lightgbm_model = LGBMRegressor(
    objective="poisson",
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)

lightgbm_model.fit(
    X_train,
    y_train,
    categorical_feature=categorical_features,
)

print("LightGBM training complete.")

In [ ]:
def lightgbm_predictor(frame):
    X_day = frame.loc[:, feature_columns]
    assert isinstance(X_day, pd.DataFrame)
    return lightgbm_model.predict(X_day)


print("Generating recursive LightGBM forecast...")
validation_forecasts["LightGBM regression"] = recursive_forecast(
    "LightGBM regression",
    lightgbm_predictor,
)

In [ ]:
validation_actuals = (
    validation_data[SERIES_KEYS + ["date", target_column]]
    .rename(columns={target_column: "actual_sales"})
)

comparison_rows = []
evaluated_forecasts = {}

for model_name, forecast in validation_forecasts.items():
    evaluated = validation_actuals.merge(
        forecast,
        on=SERIES_KEYS + ["date"],
        how="left",
        validate="one_to_one",
    )
    assert evaluated["predicted_sales"].notna().all(), (
        f"Missing predictions for {model_name}."
    )

    metrics = calculate_forecast_metrics(
        evaluated["actual_sales"],
        evaluated["predicted_sales"],
    )
    comparison_rows.append({"Model": model_name, **metrics})
    evaluated_forecasts[model_name] = evaluated

comparison_table = (
    pd.DataFrame(comparison_rows)
    .sort_values(["MAE", "RMSE", "WAPE"])
    .reset_index(drop=True)
)

comparison_table.round({"MAE": 4, "RMSE": 4, "WAPE": 2})

# Model interpretation

The fitted LightGBM model is inspected without changing its parameters or refitting it. SHAP explanations use a representative sample from the first validation day, where all series are evaluated at the same forecast information cutoff.

In [ ]:
importance_table = pd.DataFrame({
    "feature": feature_columns,
    "gain": lightgbm_model.booster_.feature_importance(importance_type="gain"),
    "split": lightgbm_model.booster_.feature_importance(importance_type="split"),
})

importance_table["gain_pct"] = (
    100 * importance_table["gain"] / importance_table["gain"].sum()
)
importance_table = importance_table.sort_values("gain", ascending=False).reset_index(drop=True)

top_n = 20
top_importance = importance_table.head(top_n).sort_values("gain")

plt.figure(figsize=(10, 7))
plt.barh(top_importance["feature"], top_importance["gain"], color="#2f6f9f")
plt.xlabel("LightGBM gain")
plt.ylabel("Feature")
plt.title(f"Top {top_n} LightGBM features by gain")
plt.tight_layout()
plt.show()

importance_table.head(top_n)

In [ ]:
shap_sample = validation_data.loc[
    validation_data["forecast_step"].eq(1),
    feature_columns,
].copy()
shap_sample = shap_sample.sample(
    n=min(2000, len(shap_sample)),
    random_state=42,
)

assert isinstance(shap_sample, pd.DataFrame)

shap_explainer = shap.TreeExplainer(lightgbm_model)
shap_values = shap_explainer.shap_values(shap_sample)
if isinstance(shap_values, list):
    shap_values = shap_values[0]

shap_values = np.asarray(shap_values)
assert shap_values.shape == shap_sample.shape

shap_importance = pd.DataFrame({
    "feature": feature_columns,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0),
})
shap_importance = shap_importance.sort_values(
    "mean_abs_shap", ascending=False
).reset_index(drop=True)

shap_importance.head(top_n)

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values,
    shap_sample,
    plot_type="bar",
    max_display=top_n,
    show=False,
)
plt.title(f"Top {top_n} validation drivers by mean absolute SHAP")
plt.tight_layout()
plt.show()

shap.summary_plot(
    shap_values,
    shap_sample,
    max_display=top_n,
    show=False,
)
plt.title("Validation SHAP summary")
plt.tight_layout()
plt.show()

In [ ]:
business_meanings = {
    "sales_lag_1": "Yesterday's demand captures immediate momentum and replenishment patterns.",
    "sales_lag_7": "Demand seven days ago captures weekly shopping seasonality.",
    "sales_lag_14": "Demand two weeks ago captures repeated biweekly patterns.",
    "sales_lag_28": "Demand four weeks ago captures monthly or four-week recurrence.",
    "sales_rolling_7_mean": "Recent weekly average represents the current demand level.",
    "sales_rolling_28_mean": "The four-week average smooths intermittent demand and trend.",
    "sales_rolling_7_std": "Recent volatility indicates how unstable daily demand is.",
    "sales_rolling_28_std": "Longer-run volatility distinguishes consistently noisy series.",
    "sell_price": "Current price captures price level and potential demand response.",
    "price_change_7": "The seven-day price change identifies promotions or price movements.",
    "price_vs_rolling_28": "Price relative to its recent norm identifies unusually high or low prices.",
    "price_available": "Price availability acts as an item-on-sale or assortment signal.",
    "day_of_week": "Day of week captures recurring weekday and weekend shopping behavior.",
    "is_weekend": "Weekend status captures the broad weekend demand uplift.",
    "month": "Month captures broad seasonal demand differences.",
    "week_of_year": "Week of year captures annual calendar seasonality.",
    "snap": "State SNAP participation can affect purchasing behavior and demand timing.",
    "has_event_1": "Calendar-event presence captures holiday or promotional demand shifts.",
}

interpretation_table = shap_importance.head(top_n).copy()
interpretation_table["business_meaning"] = interpretation_table["feature"].map(
    business_meanings
).fillna("Item, store, calendar, event, or price context used to segment demand.")

interpretation_table

# Streamlit forecast output

Inventory status is a demand-pressure recommendation because on-hand inventory and inbound orders are not available. High risk means forecast demand exceeds the recent 28-day mean by more than one standard deviation. Medium risk means forecast demand is above the recent mean. Low risk means it is at or below the recent mean.

In [ ]:
best_model_name = comparison_table.loc[
    comparison_table["MAE"].idxmin(), "Model"
]

best_forecast = evaluated_forecasts[best_model_name][
    SERIES_KEYS + ["date", "predicted_sales"]
].copy()

inventory_benchmark = (
    validation_history
    .groupby(SERIES_KEYS, observed=True)["sales"]
    .agg(
        recent_28_day_mean="mean",
        recent_28_day_std="std",
    )
    .reset_index()
)
inventory_benchmark["recent_28_day_std"] = (
    inventory_benchmark["recent_28_day_std"].fillna(0)
)

streamlit_forecast = best_forecast.merge(
    inventory_benchmark,
    on=SERIES_KEYS,
    how="left",
    validate="many_to_one",
)

high_risk_threshold = (
    streamlit_forecast["recent_28_day_mean"]
    + streamlit_forecast["recent_28_day_std"]
)
medium_risk_threshold = streamlit_forecast["recent_28_day_mean"]

streamlit_forecast["risk_level"] = np.select(
    [
        streamlit_forecast["predicted_sales"].gt(high_risk_threshold),
        streamlit_forecast["predicted_sales"].gt(medium_risk_threshold),
    ],
    ["High", "Medium"],
    default="Low",
)

streamlit_forecast["inventory_status"] = streamlit_forecast["risk_level"].map({
    "High": "Reorder now",
    "Medium": "Monitor closely",
    "Low": "Demand within recent range",
})

streamlit_forecast["inventory_recommendation"] = streamlit_forecast["risk_level"].map({
    "High": "Prioritize replenishment and review safety stock.",
    "Medium": "Review stock coverage before the forecast date.",
    "Low": "Maintain the current replenishment cadence.",
})

streamlit_forecast["predicted_sales"] = (
    streamlit_forecast["predicted_sales"].clip(lower=0).astype("float32")
)

output_columns = [
    "item_id",
    "store_id",
    "date",
    "predicted_sales",
    "inventory_status",
    "risk_level",
    "inventory_recommendation",
]
streamlit_forecast = (
    streamlit_forecast[output_columns]
    .sort_values(["date", "store_id", "item_id"])
    .reset_index(drop=True)
)

assert not streamlit_forecast.duplicated(SERIES_KEYS + ["date"]).any()
assert streamlit_forecast[output_columns].notna().all().all()
assert streamlit_forecast["date"].nunique() == FORECAST_HORIZON

print("Selected model:", best_model_name)
print("Output shape:", streamlit_forecast.shape)
streamlit_forecast.head()

In [ ]:
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
csv_output_path = OUTPUT_DIRECTORY / "streamlit_forecast_output.csv"

streamlit_forecast.to_csv(csv_output_path, index=False)

print(f"Saved CSV: {csv_output_path}")
print("Risk distribution:")
print(streamlit_forecast["risk_level"].value_counts())

# Model improvement experiments

This section preserves the current model and results as the benchmark. It builds a separate enhanced feature matrix and evaluates every candidate with the same recursive 28-day validation protocol. `improvement_vs_current` is the percentage reduction in MAE relative to the current LightGBM result of 1.3057.

The existing Streamlit output above is intentionally unchanged until an improved model is selected and reviewed.

In [ ]:
from lightgbm import LGBMClassifier

CURRENT_LIGHTGBM_METRICS = {
    "MAE": 1.3057,
    "RMSE": 2.3121,
    "WAPE": 72.55,
}

ENHANCED_HISTORY_DAYS = 56
ENHANCED_LAGS = [1, 7, 14, 28, 56]
ENHANCED_WINDOWS = [7, 14, 28, 56]
AGGREGATE_LAGS = [1, 7, 28]


def aligned_group_rolling(frame, values, window, aggregations, min_periods=None):
    groupers = [frame[column] for column in SERIES_KEYS]
    result = (
        values
        .groupby(groupers, sort=False, observed=True)
        .rolling(
            window=window,
            min_periods=window if min_periods is None else min_periods,
        )
        .agg(aggregations)
    )
    result.index = result.index.droplevel(list(range(len(SERIES_KEYS))))
    return result.reindex(frame.index)


def add_enhanced_price_features(frame):
    frame = add_price_features(frame)
    price_group = frame.groupby(SERIES_KEYS, sort=False, observed=True)["sell_price"]

    frame["price_lag_1"] = price_group.shift(1).astype("float32")
    valid_price_lag_1 = frame["price_lag_1"].replace(0, np.nan)
    frame["price_change_1"] = (
        frame["sell_price"].div(valid_price_lag_1).sub(1).astype("float32")
    )
    frame["price_discount_vs_28"] = (
        1 - frame["sell_price"].div(
            frame["price_rolling_28_mean"].replace(0, np.nan)
        )
    ).astype("float32")
    frame["promotion_flag"] = (
        frame["price_available"].eq(1)
        & frame["price_discount_vs_28"].gt(0.01)
    ).astype("int8")
    return frame


def add_aggregate_lag_features(frame):
    aggregation_levels = {
        "item": ["item_id"],
        "store": ["store_id"],
        "department": ["dept_id"],
    }

    for prefix, group_columns in aggregation_levels.items():
        daily = (
            frame
            .groupby(group_columns + ["date"], observed=True, sort=False)["sales"]
            .mean()
            .rename("aggregate_mean_sales")
            .reset_index()
            .sort_values(group_columns + ["date"])
        )
        aggregate_group = daily.groupby(
            group_columns,
            sort=False,
            observed=True,
        )["aggregate_mean_sales"]

        lookup_index = pd.MultiIndex.from_frame(frame[group_columns + ["date"]])
        for lag in AGGREGATE_LAGS:
            feature_name = f"{prefix}_mean_sales_lag_{lag}"
            daily[feature_name] = aggregate_group.shift(lag).astype("float32")
            lookup = daily.set_index(group_columns + ["date"])[feature_name]
            frame[feature_name] = lookup.reindex(lookup_index).to_numpy(dtype="float32")

    return frame


def add_enhanced_sales_features(frame):
    frame = frame.sort_values(SERIES_KEYS + ["date"]).reset_index(drop=True)
    sales_group = frame.groupby(SERIES_KEYS, sort=False, observed=True)["sales"]

    for lag in ENHANCED_LAGS:
        frame[f"sales_lag_{lag}"] = sales_group.shift(lag).astype("float32")

    shifted_sales = sales_group.shift(1)
    for window in ENHANCED_WINDOWS:
        statistics = aligned_group_rolling(
            frame,
            shifted_sales,
            window=window,
            aggregations=["mean", "std", "max"],
        )
        frame[f"sales_rolling_{window}_mean"] = statistics["mean"].astype("float32")
        frame[f"sales_rolling_{window}_std"] = statistics["std"].astype("float32")
        frame[f"sales_rolling_{window}_max"] = statistics["max"].astype("float32")

        zero_indicator = shifted_sales.eq(0).astype("float32").where(
            shifted_sales.notna()
        )
        zero_rate = aligned_group_rolling(
            frame,
            zero_indicator,
            window=window,
            aggregations=["mean"],
        )
        frame[f"sales_rolling_{window}_zero_rate"] = zero_rate["mean"].astype(
            "float32"
        )

    nonzero_indicator = shifted_sales.gt(0).astype("float32").where(
        shifted_sales.notna()
    )
    nonzero_count = aligned_group_rolling(
        frame,
        nonzero_indicator,
        window=28,
        aggregations=["sum"],
    )
    frame["nonzero_sales_count_28"] = nonzero_count["sum"].astype("float32")
    frame["sales_recent_mean_56"] = frame["sales_rolling_56_mean"]

    groupers = [frame[column] for column in SERIES_KEYS]
    expanding_mean = (
        shifted_sales
        .groupby(groupers, sort=False, observed=True)
        .expanding(min_periods=1)
        .mean()
    )
    expanding_mean.index = expanding_mean.index.droplevel(
        list(range(len(SERIES_KEYS)))
    )
    frame["sales_expanding_mean"] = (
        expanding_mean.reindex(frame.index).astype("float32")
    )

    previous_date = frame["date"].groupby(
        groupers, sort=False, observed=True
    ).shift(1)
    prior_nonzero_date = previous_date.where(shifted_sales.gt(0))
    last_nonzero_date = prior_nonzero_date.groupby(
        groupers, sort=False, observed=True
    ).ffill()
    frame["days_since_last_nonzero"] = (
        (frame["date"] - last_nonzero_date)
        .dt.days
        .clip(lower=1, upper=ENHANCED_HISTORY_DAYS)
        .fillna(ENHANCED_HISTORY_DAYS)
        .astype("float32")
    )

    return add_aggregate_lag_features(frame)


enhanced_lag_features = [f"sales_lag_{lag}" for lag in ENHANCED_LAGS]
enhanced_rolling_features = [
    f"sales_rolling_{window}_{statistic}"
    for window in ENHANCED_WINDOWS
    for statistic in ["mean", "std", "max", "zero_rate"]
]
enhanced_demand_features = [
    "days_since_last_nonzero",
    "nonzero_sales_count_28",
    "sales_recent_mean_56",
    "sales_expanding_mean",
]
enhanced_price_features = price_features + [
    "price_lag_1",
    "price_change_1",
    "price_discount_vs_28",
    "promotion_flag",
]
aggregate_features = [
    f"{prefix}_mean_sales_lag_{lag}"
    for prefix in ["item", "store", "department"]
    for lag in AGGREGATE_LAGS
]

In [ ]:
enhanced_feature_data = demand_data.copy()
enhanced_feature_data["actual_sales"] = enhanced_feature_data["sales"]
enhanced_feature_data.loc[validation_mask, "sales"] = np.nan

enhanced_feature_data = add_calendar_features(enhanced_feature_data)
enhanced_feature_data = add_enhanced_price_features(enhanced_feature_data)
enhanced_feature_data = add_enhanced_sales_features(enhanced_feature_data)

enhanced_feature_data["sales"] = enhanced_feature_data.pop("actual_sales")
enhanced_feature_data["forecast_step"] = np.where(
    enhanced_feature_data["date"].ge(validation_start),
    (enhanced_feature_data["date"] - validation_start).dt.days + 1,
    0,
).astype("int8")

enhanced_feature_columns = list(dict.fromkeys(
    categorical_features
    + calendar_features
    + enhanced_price_features
    + enhanced_lag_features
    + enhanced_rolling_features
    + enhanced_demand_features
    + aggregate_features
))

for column in categorical_features:
    enhanced_feature_data[column] = enhanced_feature_data[column].astype("category")

enhanced_train_data = enhanced_feature_data.loc[
    enhanced_feature_data["date"].lt(validation_start)
].copy()
enhanced_validation_data = enhanced_feature_data.loc[
    enhanced_feature_data["date"].ge(validation_start)
].copy()
enhanced_validation_history = (
    enhanced_train_data
    .groupby(SERIES_KEYS, sort=False, observed=True)
    .tail(ENHANCED_HISTORY_DAYS)
    .copy()
)

enhanced_X_train = enhanced_train_data.loc[:, enhanced_feature_columns]
enhanced_y_train = enhanced_train_data.loc[:, target_column]

assert isinstance(enhanced_X_train, pd.DataFrame)
assert all(
    isinstance(enhanced_X_train[column].dtype, pd.CategoricalDtype)
    for column in categorical_features
)
assert enhanced_validation_data["date"].nunique() == FORECAST_HORIZON

print("Enhanced feature count:", len(enhanced_feature_columns))
print("Enhanced training shape:", enhanced_train_data.shape)
print("Enhanced validation shape:", enhanced_validation_data.shape)

In [ ]:
def enhanced_recursive_forecast(model_name, predictor):
    history = enhanced_validation_history.copy()
    expanding_summary = (
        enhanced_train_data
        .groupby(SERIES_KEYS, sort=False, observed=True)["sales"]
        .agg(["sum", "count"])
    )
    expanding_state = {
        key: [row["sum"], row["count"]]
        for key, row in expanding_summary.iterrows()
    }
    forecast_days = sorted(enhanced_validation_data["date"].unique())
    daily_forecasts = []

    for forecast_date in forecast_days:
        current_day = enhanced_validation_data.loc[
            enhanced_validation_data["date"].eq(forecast_date)
        ].copy()
        current_day["sales"] = np.nan

        feature_frame = pd.concat(
            [history, current_day],
            ignore_index=True,
        )
        feature_frame = add_enhanced_sales_features(feature_frame)
        current_features = feature_frame.loc[
            feature_frame["date"].eq(forecast_date)
        ].copy()

        current_keys = list(
            current_features[SERIES_KEYS].itertuples(index=False, name=None)
        )
        current_features["sales_expanding_mean"] = np.asarray(
            [
                expanding_state[key][0] / expanding_state[key][1]
                if expanding_state[key][1] else np.nan
                for key in current_keys
            ],
            dtype="float32",
        )

        predictions = pd.Series(
            predictor(current_features),
            index=current_features.index,
            dtype="float64",
        )
        if len(predictions) != len(current_features):
            raise ValueError(f"{model_name} returned an unexpected number of predictions.")
        if not np.isfinite(predictions).all():
            raise ValueError(f"{model_name} returned non-finite predictions.")

        predictions = predictions.clip(lower=0)
        forecast_day = current_features[SERIES_KEYS + ["date"]].copy()
        forecast_day["predicted_sales"] = predictions.to_numpy()
        daily_forecasts.append(forecast_day)

        predicted_values = predictions.to_numpy()
        current_features["sales"] = predicted_values
        for key, prediction in zip(current_keys, predicted_values):
            expanding_state[key][0] += prediction
            expanding_state[key][1] += 1

        history = (
            pd.concat([history, current_features], ignore_index=True)
            .sort_values(SERIES_KEYS + ["date"])
            .groupby(SERIES_KEYS, sort=False, observed=True)
            .tail(ENHANCED_HISTORY_DAYS)
            .reset_index(drop=True)
        )

    forecast = pd.concat(daily_forecasts, ignore_index=True)
    forecast["Model"] = model_name
    return forecast


experiment_models = {}
experiment_forecasts = {}
experiment_rows = []


def register_experiment(model_name, model, forecast):
    evaluated = validation_actuals.merge(
        forecast,
        on=SERIES_KEYS + ["date"],
        how="left",
        validate="one_to_one",
    )
    assert evaluated["predicted_sales"].notna().all()
    metrics = calculate_forecast_metrics(
        evaluated["actual_sales"],
        evaluated["predicted_sales"],
    )
    experiment_models[model_name] = model
    experiment_forecasts[model_name] = forecast
    experiment_rows.append({"Model": model_name, **metrics})
    print(model_name, {key: round(value, 4) for key, value in metrics.items()})


BASE_EXPERIMENT_PARAMETERS = {
    "n_estimators": 400,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_child_samples": 20,
    "max_depth": -1,
    "subsample": 0.8,
    "subsample_freq": 1,
    "feature_fraction": 0.8,
    "lambda_l1": 0.0,
    "lambda_l2": 1.0,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}


def fit_single_stage_regressor(objective, parameters, variance_power=None):
    model_parameters = dict(parameters)
    model_parameters["objective"] = objective
    if objective == "tweedie":
        model_parameters["tweedie_variance_power"] = variance_power

    model = LGBMRegressor(**model_parameters)
    model.fit(
        enhanced_X_train,
        enhanced_y_train,
        categorical_feature=categorical_features,
    )
    return model


def fit_hurdle_model(parameters):
    occurrence_target = enhanced_y_train.gt(0).astype("int8")
    positive_mask = enhanced_y_train.gt(0)

    classifier_parameters = dict(parameters)
    classifier_parameters["objective"] = "binary"
    classifier = LGBMClassifier(**classifier_parameters)
    classifier.fit(
        enhanced_X_train,
        occurrence_target,
        categorical_feature=categorical_features,
    )

    positive_regressor_parameters = dict(parameters)
    positive_regressor_parameters["objective"] = "poisson"
    positive_regressor = LGBMRegressor(**positive_regressor_parameters)
    positive_regressor.fit(
        enhanced_X_train.loc[positive_mask],
        enhanced_y_train.loc[positive_mask],
        categorical_feature=categorical_features,
    )
    return classifier, positive_regressor

## Objective and hurdle screening

Poisson, three Tweedie variance powers, and the hurdle model are trained on the same enhanced feature matrix. Every candidate is evaluated recursively for all 28 validation days.

In [ ]:
objective_specs = [
    ("Enhanced Poisson", "poisson", None),
    ("Enhanced Tweedie 1.1", "tweedie", 1.1),
    ("Enhanced Tweedie 1.3", "tweedie", 1.3),
    ("Enhanced Tweedie 1.5", "tweedie", 1.5),
]

objective_lookup = {}

for model_name, objective, variance_power in objective_specs:
    print(f"Training {model_name}...")
    model = fit_single_stage_regressor(
        objective,
        BASE_EXPERIMENT_PARAMETERS,
        variance_power,
    )
    forecast = enhanced_recursive_forecast(
        model_name,
        lambda frame, fitted_model=model: fitted_model.predict(
            frame.loc[:, enhanced_feature_columns]
        ),
    )
    register_experiment(model_name, model, forecast)
    objective_lookup[model_name] = (objective, variance_power)

print("Training enhanced hurdle model...")
hurdle_classifier, hurdle_regressor = fit_hurdle_model(BASE_EXPERIMENT_PARAMETERS)


def hurdle_predictor(frame, classifier=hurdle_classifier, regressor=hurdle_regressor):
    features = frame.loc[:, enhanced_feature_columns]
    sale_probability = classifier.predict_proba(features)[:, 1]
    positive_quantity = np.clip(regressor.predict(features), 0, None)
    return sale_probability * positive_quantity


hurdle_forecast = enhanced_recursive_forecast(
    "Enhanced hurdle",
    hurdle_predictor,
)
register_experiment(
    "Enhanced hurdle",
    {"classifier": hurdle_classifier, "regressor": hurdle_regressor},
    hurdle_forecast,
)

screening_table = (
    pd.DataFrame(experiment_rows)
    .sort_values(["MAE", "RMSE", "WAPE"])
    .reset_index(drop=True)
)
screening_table.round({"MAE": 4, "RMSE": 4, "WAPE": 2})

## Small search for the leading model family

Only the best screening family is tuned. Two compact configurations vary leaves, child size, learning rate, feature fraction, and L1/L2 regularization. Each tuned candidate still receives the full recursive 28-day evaluation.

In [ ]:
best_screening_name = screening_table.loc[0, "Model"]

tuning_candidates = [
    {
        **BASE_EXPERIMENT_PARAMETERS,
        "n_estimators": 500,
        "num_leaves": 63,
        "min_child_samples": 50,
        "learning_rate": 0.04,
        "feature_fraction": 0.9,
        "lambda_l1": 0.1,
        "lambda_l2": 2.0,
    },
    {
        **BASE_EXPERIMENT_PARAMETERS,
        "n_estimators": 600,
        "num_leaves": 127,
        "min_child_samples": 100,
        "learning_rate": 0.03,
        "feature_fraction": 0.8,
        "lambda_l1": 0.5,
        "lambda_l2": 5.0,
    },
]

for candidate_number, parameters in enumerate(tuning_candidates, start=1):
    if best_screening_name == "Enhanced hurdle":
        model_name = f"Tuned hurdle {candidate_number}"
        classifier, regressor = fit_hurdle_model(parameters)

        def tuned_hurdle_predictor(
            frame,
            fitted_classifier=classifier,
            fitted_regressor=regressor,
        ):
            features = frame.loc[:, enhanced_feature_columns]
            probability = fitted_classifier.predict_proba(features)[:, 1]
            quantity = np.clip(fitted_regressor.predict(features), 0, None)
            return probability * quantity

        forecast = enhanced_recursive_forecast(
            model_name,
            tuned_hurdle_predictor,
        )
        model = {"classifier": classifier, "regressor": regressor}
    else:
        objective, variance_power = objective_lookup[best_screening_name]
        model_name = f"Tuned {best_screening_name} {candidate_number}"
        model = fit_single_stage_regressor(
            objective,
            parameters,
            variance_power,
        )
        forecast = enhanced_recursive_forecast(
            model_name,
            lambda frame, fitted_model=model: fitted_model.predict(
                frame.loc[:, enhanced_feature_columns]
            ),
        )

    register_experiment(model_name, model, forecast)

In [ ]:
current_benchmark_row = {
    "Model": "Current LightGBM benchmark",
    **CURRENT_LIGHTGBM_METRICS,
}

experiment_comparison_table = pd.concat(
    [
        pd.DataFrame([current_benchmark_row]),
        pd.DataFrame(experiment_rows),
    ],
    ignore_index=True,
)
experiment_comparison_table["improvement_vs_current"] = (
    100
    * (CURRENT_LIGHTGBM_METRICS["MAE"] - experiment_comparison_table["MAE"])
    / CURRENT_LIGHTGBM_METRICS["MAE"]
)
experiment_comparison_table = (
    experiment_comparison_table
    .sort_values(["MAE", "RMSE", "WAPE"])
    .reset_index(drop=True)
)

best_experiment_name = experiment_comparison_table.loc[0, "Model"]
if best_experiment_name == "Current LightGBM benchmark":
    best_experiment_model = lightgbm_model
    best_experiment_forecast = validation_forecasts["LightGBM regression"]
else:
    best_experiment_model = experiment_models[best_experiment_name]
    best_experiment_forecast = experiment_forecasts[best_experiment_name]

print("Best model:", best_experiment_name)
experiment_comparison_table.round({
    "MAE": 4,
    "RMSE": 4,
    "WAPE": 2,
    "improvement_vs_current": 2,
})

# Direct multi-horizon experiment

This final experiment trains one global Poisson LightGBM for horizons 1-28. Historical training dates are divided into non-overlapping 28-day blocks. Within each block, every target uses demand-history features frozen at the block's original forecast cutoff, while known target-date calendar, event, SNAP, and price features remain horizon-specific. Validation uses one cutoff snapshot for all 28 horizons, with no validation sales or recursive predictions used as features.

`M5_DIRECT_TRAINING_BLOCKS` controls the number of recent 28-day blocks used for training and defaults to 18 to keep Kaggle memory and runtime practical.

In [ ]:
DIRECT_TRAINING_BLOCKS = int(os.environ.get("M5_DIRECT_TRAINING_BLOCKS", "18"))
if DIRECT_TRAINING_BLOCKS < 1:
    raise ValueError("M5_DIRECT_TRAINING_BLOCKS must be at least 1.")

direct_history_features = list(dict.fromkeys(
    enhanced_lag_features
    + enhanced_rolling_features
    + enhanced_demand_features
    + aggregate_features
))
direct_target_features = list(dict.fromkeys(
    categorical_features + calendar_features + enhanced_price_features
))
direct_feature_columns = list(dict.fromkeys(
    enhanced_feature_columns + ["forecast_horizon"]
))

# Align complete 28-day training blocks backwards from the validation cutoff.
days_before_validation = (
    validation_start - enhanced_train_data["date"]
).dt.days
training_horizon = (
    28 - ((days_before_validation - 1) % FORECAST_HORIZON)
).astype("int8")
training_block = (
    (days_before_validation - 1) // FORECAST_HORIZON
).astype("int16")
direct_training_mask = (
    days_before_validation.ge(1)
    & training_block.ge(0)
    & training_block.lt(DIRECT_TRAINING_BLOCKS)
)

direct_target_columns = list(dict.fromkeys(
    SERIES_KEYS
    + ["date", target_column]
    + direct_target_features
))
direct_targets = enhanced_train_data.loc[
    direct_training_mask,
    direct_target_columns,
].copy()
direct_targets["forecast_horizon"] = training_horizon.loc[
    direct_training_mask
].to_numpy()

# The feature date is cutoff + 1. Its shifted demand features contain sales
# through the cutoff and are shared by every horizon in the same block.
direct_targets["history_feature_date"] = (
    direct_targets["date"]
    - pd.to_timedelta(direct_targets["forecast_horizon"] - 1, unit="D")
)
history_feature_dates = direct_targets["history_feature_date"].unique()
direct_history_lookup = enhanced_train_data.loc[
    enhanced_train_data["date"].isin(history_feature_dates),
    SERIES_KEYS + ["date"] + direct_history_features,
].rename(columns={"date": "history_feature_date"})

direct_train_data = direct_targets.merge(
    direct_history_lookup,
    on=SERIES_KEYS + ["history_feature_date"],
    how="inner",
    validate="many_to_one",
)

assert len(direct_train_data) == len(direct_targets), (
    "Some direct-training rows could not be matched to cutoff features."
)
assert target_column not in direct_feature_columns
assert set(direct_feature_columns).issubset(direct_train_data.columns)
assert direct_train_data["forecast_horizon"].between(1, FORECAST_HORIZON).all()
assert all(
    isinstance(direct_train_data[column].dtype, pd.CategoricalDtype)
    for column in categorical_features
)

direct_training_rows = len(direct_train_data)
print("Direct training blocks:", DIRECT_TRAINING_BLOCKS)
print("Direct training rows:", f"{direct_training_rows:,}")
print("Direct feature count:", len(direct_feature_columns))

del direct_targets, direct_history_lookup

In [ ]:
import gc

direct_model_parameters = {
    **BASE_EXPERIMENT_PARAMETERS,
    "objective": "poisson",
}
direct_lightgbm_model = LGBMRegressor(**direct_model_parameters)
direct_lightgbm_model.fit(
    direct_train_data.loc[:, direct_feature_columns],
    direct_train_data.loc[:, target_column],
    categorical_feature=categorical_features,
)

# Build all validation horizons from the same original cutoff snapshot.
direct_validation_targets = enhanced_validation_data.loc[
    :,
    list(dict.fromkeys(SERIES_KEYS + ["date"] + direct_target_features)),
].copy()
direct_validation_targets["forecast_horizon"] = (
    (direct_validation_targets["date"] - validation_start).dt.days + 1
).astype("int8")

direct_cutoff_history = enhanced_validation_data.loc[
    enhanced_validation_data["date"].eq(validation_start),
    SERIES_KEYS + direct_history_features,
].copy()
assert not direct_cutoff_history.duplicated(SERIES_KEYS).any()

direct_validation_features = direct_validation_targets.merge(
    direct_cutoff_history,
    on=SERIES_KEYS,
    how="left",
    validate="many_to_one",
)
assert len(direct_validation_features) == len(enhanced_validation_data)
assert set(direct_feature_columns).issubset(direct_validation_features.columns)
assert all(
    isinstance(direct_validation_features[column].dtype, pd.CategoricalDtype)
    for column in categorical_features
)

direct_predictions = np.clip(
    direct_lightgbm_model.predict(
        direct_validation_features.loc[:, direct_feature_columns]
    ),
    0,
    None,
)
direct_forecast = direct_validation_features[
    SERIES_KEYS + ["date", "forecast_horizon"]
].copy()
direct_forecast["predicted_sales"] = direct_predictions
direct_forecast["Model"] = "Direct multi-horizon Poisson"

expected_direct_rows = (
    demand_data[SERIES_KEYS].drop_duplicates().shape[0] * FORECAST_HORIZON
)
assert len(direct_forecast) == expected_direct_rows
assert direct_forecast["date"].nunique() == FORECAST_HORIZON
assert not direct_forecast.duplicated(SERIES_KEYS + ["date"]).any()
assert np.isfinite(direct_forecast["predicted_sales"]).all()

# Release the expanded historical design matrix before evaluation/export.
del direct_train_data
gc.collect()

print("Direct model training and 28-day forecast complete.")

In [ ]:
direct_evaluated = validation_actuals.merge(
    direct_forecast,
    on=SERIES_KEYS + ["date"],
    how="left",
    validate="one_to_one",
)
direct_metrics = calculate_forecast_metrics(
    direct_evaluated["actual_sales"],
    direct_evaluated["predicted_sales"],
)

enhanced_poisson_metrics = next(
    {
        "MAE": row["MAE"],
        "RMSE": row["RMSE"],
        "WAPE": row["WAPE"],
    }
    for row in experiment_rows
    if row["Model"] == "Enhanced Poisson"
)

direct_vs_poisson_table = pd.DataFrame([
    {"Model": "Enhanced Poisson", **enhanced_poisson_metrics},
    {"Model": "Direct multi-horizon Poisson", **direct_metrics},
])
direct_vs_poisson_table["MAE_change_vs_enhanced_poisson"] = (
    direct_vs_poisson_table["MAE"] - enhanced_poisson_metrics["MAE"]
)
direct_improves_enhanced_poisson = (
    direct_metrics["MAE"] < enhanced_poisson_metrics["MAE"]
)

horizon_metric_rows = []
for horizon, horizon_frame in direct_evaluated.groupby(
    "forecast_horizon", sort=True
):
    horizon_metric_rows.append({
        "forecast_horizon": int(horizon),
        **calculate_forecast_metrics(
            horizon_frame["actual_sales"],
            horizon_frame["predicted_sales"],
        ),
    })
direct_metrics_by_horizon = pd.DataFrame(horizon_metric_rows)

print(
    "Direct forecasting improves Enhanced Poisson on validation MAE:",
    direct_improves_enhanced_poisson,
)
display(direct_vs_poisson_table.round({
    "MAE": 4,
    "RMSE": 4,
    "WAPE": 2,
    "MAE_change_vs_enhanced_poisson": 4,
}))
display(direct_metrics_by_horizon.round({
    "MAE": 4,
    "RMSE": 4,
    "WAPE": 2,
}))

# Final validated model selection and Streamlit export

The final export selects the lowest-validation-MAE model from the current LightGBM benchmark, Enhanced Poisson, and the direct multi-horizon model. It then applies the existing inventory status, risk, and recommendation rules without changing the dashboard schema.

In [ ]:
final_model_comparison = pd.DataFrame([
    {
        "Model": "Current LightGBM benchmark",
        **CURRENT_LIGHTGBM_METRICS,
    },
    {
        "Model": "Enhanced Poisson",
        **enhanced_poisson_metrics,
    },
    {
        "Model": "Direct multi-horizon Poisson",
        **direct_metrics,
    },
]).sort_values(["MAE", "RMSE", "WAPE"]).reset_index(drop=True)

final_selected_model = final_model_comparison.loc[0, "Model"]
final_forecast_lookup = {
    "Current LightGBM benchmark": validation_forecasts["LightGBM regression"],
    "Enhanced Poisson": experiment_forecasts["Enhanced Poisson"],
    "Direct multi-horizon Poisson": direct_forecast,
}
final_selected_forecast = final_forecast_lookup[final_selected_model][
    SERIES_KEYS + ["date", "predicted_sales"]
].copy()

final_streamlit_forecast = final_selected_forecast.merge(
    inventory_benchmark,
    on=SERIES_KEYS,
    how="left",
    validate="many_to_one",
)
final_high_risk_threshold = (
    final_streamlit_forecast["recent_28_day_mean"]
    + final_streamlit_forecast["recent_28_day_std"]
)
final_medium_risk_threshold = final_streamlit_forecast["recent_28_day_mean"]

final_streamlit_forecast["risk_level"] = np.select(
    [
        final_streamlit_forecast["predicted_sales"].gt(
            final_high_risk_threshold
        ),
        final_streamlit_forecast["predicted_sales"].gt(
            final_medium_risk_threshold
        ),
    ],
    ["High", "Medium"],
    default="Low",
)
final_streamlit_forecast["inventory_status"] = (
    final_streamlit_forecast["risk_level"].map({
        "High": "Reorder now",
        "Medium": "Monitor closely",
        "Low": "Demand within recent range",
    })
)
final_streamlit_forecast["inventory_recommendation"] = (
    final_streamlit_forecast["risk_level"].map({
        "High": "Prioritize replenishment and review safety stock.",
        "Medium": "Review stock coverage before the forecast date.",
        "Low": "Maintain the current replenishment cadence.",
    })
)
final_streamlit_forecast["predicted_sales"] = (
    final_streamlit_forecast["predicted_sales"]
    .clip(lower=0)
    .astype("float32")
)
final_streamlit_forecast = (
    final_streamlit_forecast[output_columns]
    .sort_values(["date", "store_id", "item_id"])
    .reset_index(drop=True)
)

assert len(final_streamlit_forecast) == expected_direct_rows
assert final_streamlit_forecast["date"].nunique() == FORECAST_HORIZON
assert not final_streamlit_forecast.duplicated(
    SERIES_KEYS + ["date"]
).any()
assert final_streamlit_forecast[output_columns].notna().all().all()

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
final_csv_output_path = OUTPUT_DIRECTORY / "streamlit_forecast_output.csv"
final_streamlit_forecast.to_csv(final_csv_output_path, index=False)

print("Final selected model:", final_selected_model)
print("Saved final CSV:", final_csv_output_path)
display(final_model_comparison.round({
    "MAE": 4,
    "RMSE": 4,
    "WAPE": 2,
}))
final_streamlit_forecast.head()